In [1]:
import pandas as pd

orders = pd.read_csv('olist_orders_dataset.csv', dtype='string')
print('Rows and columns:', orders.shape)
display(orders.head(3))

Rows and columns: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [2]:
print('Repeated order IDs:', orders['order_id'].duplicated().sum())
print('Missing values per column:')
print(orders.isna().sum())

Repeated order IDs: 0
Missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [3]:
missing_delivery = orders['order_delivered_customer_date'].isna()
print('All orders by status:')
print(orders['order_status'].value_counts(dropna=False))
print('Orders missing a customer delivery date, by status:')
print(orders.loc[missing_delivery, 'order_status'].value_counts(dropna=False))

All orders by status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: Int64
Orders missing a customer delivery date, by status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: Int64


In [4]:
actual_delivery = pd.to_datetime(orders['order_delivered_customer_date'], errors='raise')
estimated_delivery = pd.to_datetime(orders['order_estimated_delivery_date'], errors='raise')
print('Actual delivery type:', actual_delivery.dtype)
print('Estimated delivery type:', estimated_delivery.dtype)


Actual delivery type: datetime64[us]
Estimated delivery type: datetime64[us]


In [5]:
eligible = (
    (orders['order_status'] == 'delivered')
    & actual_delivery.notna()
    & estimated_delivery.notna()
)
print('Delivered orders with both dates:', eligible.sum())

Delivered orders with both dates: 96470


In [6]:
late = eligible & (actual_delivery.dt.normalize() > estimated_delivery.dt.normalize())
late_rate = late.sum() / eligible.sum() * 100
print('Late orders:', late.sum())
print('Late-delivery rate (%):', round(late_rate, 2))

Late orders: 6534
Late-delivery rate (%): 6.77


In [8]:
delay_days = (actual_delivery.dt.normalize() - estimated_delivery.dt.normalize()).dt.days
late_delay_days = delay_days.loc[late]
print('Median days late:', late_delay_days.median())
print('Average days late:', round(late_delay_days.mean(), 2))

Median days late: 7.0
Average days late: 10.62


In [9]:
purchase_time = pd.to_datetime(orders['order_purchase_timestamp'], errors='raise')
delivery_before_purchase = actual_delivery.notna() & (actual_delivery < purchase_time)
print('Missing purchase timestamps:', purchase_time.isna().sum())
print('Delivery timestamps before purchase:', delivery_before_purchase.sum())

Missing purchase timestamps: 0
Delivery timestamps before purchase: 0
